# Process Tables with Auto Loader

This notebook runs after the For Each download tasks complete.
It queries the audit table to find successful downloads and processes them into Delta tables using Auto Loader.

**Workflow:**
1. Query audit table for successful downloads
2. Get unique list of tables with data
3. Call autoloader_process for each table
4. Generate summary report

In [0]:
import json
from datetime import datetime

In [0]:
# Widget parameters
dbutils.widgets.text("catalog", "sandbox", "Catalog name")
dbutils.widgets.text("schema", "cms_data_raw", "Schema name")
dbutils.widgets.text("mode", "incremental", "Mode: backfill or incremental")

In [0]:
# Get parameters
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
mode = dbutils.widgets.get("mode")

print(f"=== Process Tables with Auto Loader ===")
print(f"Mode: {mode}")
print(f"Target: {catalog}.{schema}")
print()

In [0]:
# Load table mappings
with open("/Workspace/Users/cvollstadt@gmail.com/cms_data_load/src/table_mappings.json", "r") as f:
    config = json.load(f)
    mappings = config["mappings"]

print(f"✓ Loaded {len(mappings)} table mappings")

In [0]:
# Query audit table to find successful downloads
audit_table = f"{catalog}.{schema}.cms_pipeline_audit"

print(f"\nQuerying audit table: {audit_table}")

# Get successful downloads from recent run
successful_downloads_df = spark.sql(f"""
    SELECT DISTINCT 
        landing_path,
        catalog,
        schema
    FROM {audit_table}
    WHERE download_status = 'success'
        AND run_timestamp >= current_timestamp() - INTERVAL 1 DAY
    ORDER BY landing_path
""")

successful_tables = [row.landing_path for row in successful_downloads_df.collect()]

print(f"\n✓ Found {len(successful_tables)} tables with successful downloads")
for table in successful_tables[:5]:
    print(f"  - {table}")
if len(successful_tables) > 5:
    print(f"  ... and {len(successful_tables) - 5} more")

In [0]:
# Process each table with Auto Loader
print(f"\n=== Processing {len(successful_tables)} tables with Auto Loader ===\n")

processing_results = []

for i, landing_path in enumerate(successful_tables, 1):
    # Find table name and comment from mappings
    table_name = next((m["table_name"] for m in mappings if m["landing_path"] == landing_path), landing_path)
    table_comment = next((m["comment"] for m in mappings if m["landing_path"] == landing_path), "")
    
    source_path = f"/Volumes/{catalog}/{schema}/cms_landing/{landing_path}"
    
    print(f"[{i}/{len(successful_tables)}] Processing {table_name}...")
    
    try:
        params = {
            "source_path": source_path,
            "table_name": table_name,
            "catalog": catalog,
            "schema": schema,
            "checkpoint_path": "",  # Will auto-generate
            "comment": table_comment
        }
        
        result = dbutils.notebook.run(
            "./autoloader_process",
            timeout_seconds=1800,  # 30 minute timeout per table
            arguments=params
        )
        
        result_dict = json.loads(result) if result else {}
        processing_results.append({
            "table_name": table_name,
            "result": result_dict,
            "success": True
        })
        
        total_rows = result_dict.get("total_rows", 0)
        print(f"  ✓ Loaded {total_rows:,} rows")
        
    except Exception as e:
        processing_results.append({
            "table_name": table_name,
            "error": str(e),
            "success": False
        })
        print(f"  ✗ Error: {str(e)}")

In [0]:
# Generate summary
successful_processing = sum(1 for r in processing_results if r["success"])
failed_processing = len(processing_results) - successful_processing
total_rows_loaded = sum(r["result"].get("total_rows", 0) for r in processing_results if r["success"])

summary = {
    "mode": mode,
    "timestamp": datetime.now().isoformat(),
    "processing": {
        "tables_processed": len(successful_tables),
        "successful": successful_processing,
        "failed": failed_processing,
        "total_rows_loaded": total_rows_loaded
    },
    "configuration": {
        "catalog": catalog,
        "schema": schema
    }
}

print("\n" + "="*60)
print("=== PROCESSING SUMMARY ===")
print("="*60)
print(f"\nMode: {mode}")
print(f"\nProcessing:")
print(f"  Tables processed: {len(successful_tables)}")
print(f"  ✓ Successful: {successful_processing}")
print(f"  ✗ Failed: {failed_processing}")
print(f"  Total rows loaded: {total_rows_loaded:,}")
print(f"\nTarget: {catalog}.{schema}")
print("="*60)

In [0]:
# Return summary
import json
dbutils.notebook.exit(json.dumps(summary))